# Faruq-v3 DSRDet/FBNR transfer breadth screening

Runs two seed-42 training-only regularization candidates from the same frozen D0 checkpoint: **FGC1** (structure-agnostic Gaussian foreground concealment) and **FBR1** (stochastic foreground/background decoupled regularization). The transfer keeps YOLO26 inference architecture unchanged. It is not a literal reproduction of DSRDet's aircraft cross-shape prior or gradient-domain Poisson blending. Test is never extracted or opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO=Path('/content/coffee-bean-detection')
BRANCH='agent/dsr-fbnr-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    r=subprocess.run(clone)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
os.chdir(REPO)
print('REPO:',REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan GPU.'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'
assert GROUPED.is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-fbnr-transfer-screening-v1'
print('GPU:',torch.cuda.get_device_name(0))
print('OUTPUT:',OUTPUT)

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_fbnr_screening',
 '--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),'--control-summary',str(CONTROL),
 '--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('MENJALANKAN:',' '.join(command),flush=True)
p=subprocess.run(command,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(p.stdout,end='',flush=True)
if p.returncode!=0: raise RuntimeError(f'FBNR screening gagal: {p.returncode}')

In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/fbnr_seed42_screening.json'
result=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['test_opened'] is False and result['test_images_accessed'] is False
rows=[]
for name,m in result['controls'].items(): rows.append({'model':name,**m,'decision':'control'})
for name,p in result['candidates'].items(): rows.append({'model':name,**p['metrics'],'decision':result['decisions'][name]['decision']})
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
print('FGC1:',result['decisions']['FGC1'])
print('FBR1:',result['decisions']['FBR1'])
print('SUMMARY:',SUMMARY)
print('Jangan membuka test atau menjalankan seed lain dari notebook discovery ini.')